In [2]:
!pip -q install sqlalchemy psycopg2-binary pandas

from google.colab import files
import pandas as pd
from sqlalchemy import create_engine, text
from getpass import getpass

# Upload Online Retail CSV
uploaded = files.upload()

print("Uploaded files:")
for file_name in uploaded.keys():
    print("-", file_name)

csv_file = list(uploaded.keys())[0]

# Read CSV
df = pd.read_csv(
    csv_file,
    encoding="ISO-8859-1",
    dtype={
        "Invoice": "string",
        "StockCode": "string",
        "Description": "string",
        "Customer ID": "string",
        "Country": "string"
    }
)

print("Original shape:", df.shape)
print("Original columns:")
print(df.columns.tolist())

# Rename columns to match raw.online_retail_ii table in Neon
df = df.rename(columns={
    "Invoice": "invoice",
    "StockCode": "stock_code",
    "Description": "description",
    "Quantity": "quantity",
    "InvoiceDate": "invoice_date_text",
    "Price": "price",
    "Customer ID": "customer_id",
    "Country": "country"
})

# Keep only required columns in correct order
df = df[
    [
        "invoice",
        "stock_code",
        "description",
        "quantity",
        "invoice_date_text",
        "price",
        "customer_id",
        "country"
    ]
]

print("Cleaned raw-upload columns:")
print(df.columns.tolist())

print("Preview:")
display(df.head())

# Connect to Neon
connection_string = getpass("Paste your Neon connection string here: ")
engine = create_engine(connection_string)

# Clear existing raw table to avoid duplicate rows if rerun
with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE raw.online_retail_ii;"))

# Upload to Neon
df.to_sql(
    name="online_retail_ii",
    con=engine,
    schema="raw",
    if_exists="append",
    index=False,
    chunksize=5000,
    method="multi"
)

# Validate uploaded row count
with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM raw.online_retail_ii;"))
    uploaded_rows = result.scalar()

print(f"Rows uploaded to raw.online_retail_ii: {uploaded_rows}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 14.6 MB/s eta 0:00:00


Saving online_retail_II.csv to online_retail_II.csv
Uploaded files:
- online_retail_II.csv
Original shape: (541910, 8)
Original columns:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']
Cleaned raw-upload columns:
['invoice', 'stock_code', 'description', 'quantity', 'invoice_date_text', 'price', 'customer_id', 'country']
Preview:


,invoice,stock_code,description,quantity,invoice_date_text,price,customer_id,country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/10 8:26,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/10 8:26,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/10 8:26,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/10 8:26,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/10 8:26,3.39,17850,United Kingdom


Paste your Neon connection string here: ··········
Rows uploaded to raw.online_retail_ii: 541910


In [3]:
!pip -q install sqlalchemy psycopg2-binary pandas

import os
import zipfile
import pandas as pd
from sqlalchemy import create_engine
from getpass import getpass
from google.colab import files

# Connect to Neon safely
connection_string = getpass("Paste your Neon connection string here: ")
engine = create_engine(connection_string)

# Export folder
export_folder = "online_retail_tableau_exports"
os.makedirs(export_folder, exist_ok=True)

# Tableau-ready mart views
views = [
    "vw_dashboard1_rfm_customers",
    "vw_dashboard1_rfm_segment_summary",
    "vw_dashboard2_cohort_retention",
    "vw_dashboard2_retention_curve",
    "vw_dashboard3_product_performance",
    "vw_dashboard3_monthly_revenue_returns",
    "vw_dashboard3_product_pairs"
]

# Export each view as CSV
for view in views:
    query = f"SELECT * FROM mart.{view};"
    df_export = pd.read_sql(query, engine)

    output_path = os.path.join(export_folder, f"{view}.csv")
    df_export.to_csv(output_path, index=False)

    print(f"Exported {view}: {df_export.shape[0]} rows, {df_export.shape[1]} columns")

# Zip all exported CSV files
zip_name = "online_retail_tableau_exports.zip"

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file_name in os.listdir(export_folder):
        file_path = os.path.join(export_folder, file_name)
        zipf.write(file_path, arcname=file_name)

files.download(zip_name)

print("Export completed:", zip_name)

Paste your Neon connection string here: ··········
Exported vw_dashboard1_rfm_customers: 4338 rows, 10 columns
Exported vw_dashboard1_rfm_segment_summary: 9 rows, 6 columns
Exported vw_dashboard2_cohort_retention: 91 rows, 6 columns
Exported vw_dashboard2_retention_curve: 13 rows, 4 columns
Exported vw_dashboard3_product_performance: 4582 rows, 13 columns
Exported vw_dashboard3_monthly_revenue_returns: 13 rows, 6 columns
Exported vw_dashboard3_product_pairs: 101 rows, 6 columns


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Export completed: online_retail_tableau_exports.zip


In [1]:
import pandas as pd
from sqlalchemy import create_engine
from getpass import getpass
from google.colab import files

connection_string = getpass("Paste your Neon connection string here: ")
engine = create_engine(connection_string)

df_pareto = pd.read_sql(
    "SELECT * FROM mart.vw_dashboard3_product_pareto_top20 ORDER BY pareto_rank;",
    engine
)

output_file = "vw_dashboard3_product_pareto_top20.csv"
df_pareto.to_csv(output_file, index=False)

files.download(output_file)

print("Exported:", output_file)
print(df_pareto.shape)

Paste your Neon connection string here: ··········


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Exported: vw_dashboard3_product_pareto_top20.csv
(20, 12)


In [2]:
import pandas as pd
from sqlalchemy import create_engine
from getpass import getpass
from google.colab import files

connection_string = getpass("Paste your Neon connection string here: ")
engine = create_engine(connection_string)

df_return_risk = pd.read_sql(
    "SELECT * FROM mart.vw_dashboard3_product_return_risk_top20 ORDER BY return_risk_rank;",
    engine
)

output_file = "vw_dashboard3_product_return_risk_top20.csv"
df_return_risk.to_csv(output_file, index=False)

files.download(output_file)

print("Exported:", output_file)
print(df_return_risk.shape)

Paste your Neon connection string here: ··········


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Exported: vw_dashboard3_product_return_risk_top20.csv
(20, 11)


In [3]:
import pandas as pd
from sqlalchemy import create_engine
from getpass import getpass
from google.colab import files

connection_string = getpass("Paste your Neon connection string here: ")
engine = create_engine(connection_string)

df_return_risk = pd.read_sql(
    """
    SELECT *
    FROM mart.vw_dashboard3_product_return_risk_top20
    ORDER BY return_risk_rank;
    """,
    engine
)

output_file = "vw_dashboard3_product_return_risk_top20_corrected.csv"
df_return_risk.to_csv(output_file, index=False)

files.download(output_file)

print("Exported:", output_file)
print(df_return_risk.shape)
df_return_risk.head()

Paste your Neon connection string here: ··········


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Exported: vw_dashboard3_product_return_risk_top20_corrected.csv
(20, 11)


,return_risk_rank,product_label,stock_code,product_description,sales_quantity,sales_revenue,sales_orders,returned_quantity,returned_value,return_orders,return_share_pct
0,1,001. 22618 - COOKING SET RETROSPOT,22618,COOKING SET RETROSPOT,66,621.66,25,1701,174.18,4,96.26
1,2,002. 23114 - VINTAGE LEAF CHOPPING BOARD,23114,VINTAGE LEAF CHOPPING BOARD,144,684.00,41,1443,14.85,3,90.93
2,3,003. 23115 - RED APPLES CHOPPING BOARD,23115,RED APPLES CHOPPING BOARD,147,698.85,38,1440,0.00,1,90.74
3,4,004. 23005 - TRAVEL CARD WALLET I LOVE LONDON,23005,TRAVEL CARD WALLET I LOVE LONDON,4396,1788.72,175,19201,0.42,3,81.37
4,5,005. 85036B - CHOCOLATE 1 WICK MORRIS BOX CANDLE,85036B,CHOCOLATE 1 WICK MORRIS BOX CANDLE,360,581.58,42,1313,25.50,3,78.48


In [4]:
import pandas as pd
from sqlalchemy import create_engine
from getpass import getpass
from google.colab import files

connection_string = getpass("Paste your Neon connection string here: ")
engine = create_engine(connection_string)

df_kpi = pd.read_sql(
    """
    SELECT *
    FROM mart.vw_dashboard3_kpi_summary;
    """,
    engine
)

output_file = "vw_dashboard3_kpi_summary.csv"
df_kpi.to_csv(output_file, index=False)

files.download(output_file)

print("Exported:", output_file)
print(df_kpi.shape)
df_kpi.head()

Paste your Neon connection string here: ··········


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Exported: vw_dashboard3_kpi_summary.csv
(1, 6)


,active_products,total_revenue,total_countries,weighted_return_share_pct,products_needed_for_80pct_revenue,top20_revenue_share_pct
0,3665,8911425.9,37,8.12,777,14.37
